In [1]:
import os
import shutil
import xml.etree.ElementTree as ET
import random
import cv2
import matplotlib.pyplot as plt
from PIL import Image
from skimage import io
import numpy as np
from random import sample
import copy

In [2]:
class SetAnnotation():
    def __init__(self,in_file_xml,in_dir_img,out_dir_xml,classes=['Pest', 'Spider', 'Fly', 'Snail','Aphids','Slug','Beetle']):
        # assert(in_dir_xml!=out_dir_xml)
        self.in_file_xml = in_file_xml
        self.in_dir_img = in_dir_img
        self.out_dir_xml = out_dir_xml
        self.classes = classes

    def __call__(self,image_name,imagesize,pred):
        global data_root
        #确定文件的输入输出
        filepath = f'{self.in_dir_img}/{image_name}.jpg'
        in_file = open(self.in_file_xml, encoding='utf-8')
        out_file = f'{self.out_dir_xml}/{image_name}.xml'
        #获取文件的层次结构
        tree=ET.parse(in_file)
        root = tree.getroot()  #获取root节点，即1.3中的annotation节点
        folder = root.find('folder')
        folder.text = self.in_dir_img
        filename = root.find('filename')
        filename.text = image_name
        path = root.find('path')
        path.text = filepath
        size = root.find('size') #获取size节点
        size.find('width').text = f'{imagesize[0]}'#获取宽度节点信息
        size.find('height').text=  f'{imagesize[1]}'#获取高度节点信息
        if len(imagesize) == 3:
            size.find('depth').text =  f'{imagesize[2]}'
        objects = root.find('object')

        if pred.shape[0] == 0:
            return

        nums_new_object = pred.shape[0] - len(root.findall('object'))

        if nums_new_object > 0:
            for i in range(nums_new_object):
                # Create a copy
                obj = copy.deepcopy(objects)
                # Append the copy
                root.append(obj)
        elif nums_new_object < 0:
            for obj in root.findall('object'):
                root.remove(obj)
                nums_new_object+=1
                if nums_new_object==0:
                    break

        for i, obj in enumerate(root.iter('object')):  #迭代获取所有的object节点
            print(pred[i])
            cls = self.classes[int(pred[i][5])]
            obj.find('name').text = cls          #获取name节点信息，即bbox的类别信息
            xmlbox = obj.find('bndbox')          #获取bbox的左上角点与右下角点坐标信息
            if int(pred[i][0]) != 0:
                xmlbox.find('xmin').text = f'{int(pred[i][0])}'
                xmlbox.find('ymin').text = f'{int(pred[i][1])}'
                xmlbox.find('xmax').text = f'{int(pred[i][2])}'
                xmlbox.find('ymax').text = f'{int(pred[i][3])}'
            else:
                xmlbox.find('xmin').text = f'{int(pred[i][0]* imagesize[0]) }'
                xmlbox.find('ymin').text = f'{int(pred[i][1]* imagesize[1]) }'
                xmlbox.find('xmax').text = f'{int(pred[i][2]* imagesize[0]) }'
                xmlbox.find('ymax').text = f'{int(pred[i][3]* imagesize[1]) }'
        # print(out_file)
        tree.write(out_file,encoding='utf-8')

In [3]:
category_map = {
  'INSECTA': 0, 
  'FROGHOPPER (CERCOPIDAE)': 0, 
  'SCARABAEIDAE': 0,
  'GRAIN APHID': 1,
  'GRAIN APHID (SITOBION AVENAE)': 1,
  '11121': 1,
  'ROSE GRAIN APHID': 1,
  'BIRD CHERRY OAT APHID': 2,
  'BIRD CHERRY OAK APHIDS': 2,
  '11122': 2,
  'WILLOW CARROT APHID': 3,
  'WILLOW-CARROT APHIDS': 3,
  'POLLEN BEETLE (MELIGETHES SPP.)': 4, 
  'SNAIL': 5, 
  'CEREAL LEAF BEETLE (OULEMA MELANOPUS)': 6, 
  'FLY (DIPTERA)': 7, 
  'FLY (MELANOSTOMA SPP.)': 7, 
  'BIBIONID FLY (BIBIONIDAE)': 7, 
  'MUSCIDAE (FLY)': 7, 
  'BEAN SEED FLY (DELIA SPP.)': 7, 
  'FUNGUS GNAT (MYCETOPHILIDAE)': 7, 
  'LEAF MINERS': 7,  
  'CABBAGE STEM FLEA BETTLE': 8, 
  'LADYBUG (COCCINELLIDAE)': 9, 
  'LADYBUG (COCCINELLIDAE) (PUPA)': 10, 
  'LADYBUG (COCCINELLIDAE) (LARVAE)': 11, 
  'SPIDER (ARANEUS SPP.)': 12, 
  'CHIRONOMID MIDGE': 13, 'CHIRONOMID MIDGE (MALE)': 13,
  'BEETLE (COLEOPTERA)': 14, 
  'MOSQUITO': 15,
  'WASP': 16, 
  'BEE': 17,
  'SLUG': 18, 
  'CABBAGE WHITEFLY': 19,
  'HEMIPTERA (PLANT BUG)': 20,
  'EARTHWORM': 21, 
  'BUMBLEBEE': 22, 
  'GROUND BETTLE (HARPALUS SPP)': 23, 
  'ANT': 24, 
  'PYRRHOCORIDAE': 25, 
  'LONGICORN': 26,
  '11111': 0,
}

In [4]:
def convert(size,box):
    x = (box[0] + box[2])/2.0 - 1
    y = (box[1] + box[3])/2.0 - 1
    x = x/float(size[0])
    y = y/float(size[1])

    w = (box[2] - box[0])/float(size[0])
    h = (box[3] - box[1])/float(size[1])

    return x,y,w,h

def convert_annotation(
    annotation_file_path, 
    list_file, 
    classes_dict:dict,
    img_file_path=None):
    
    voc_annotation_file = open(annotation_file_path, encoding='utf-8')
    tree = ET.parse(voc_annotation_file)
    root = tree.getroot()

    width = float(root.find('size')[0].text)
    height = float(root.find('size')[1].text)

    if img_file_path:
        img = io.imread(img_file_path)
        if img.shape[1] != width or img.shape[0] != height:
            print(f'Image size does not match annotation size, {img_file_path}')
            width = img.shape[1]
            height = img.shape[0]
            print(img.shape)

    for obj in root.iter('object'):
        class_name = obj.find('name').text.upper()
                    
        if not class_name in category_map:
            print(class_name)
            class_name = "INSECTA"

        cls_id = category_map[class_name]

        if cls_id in classes_dict:
            if class_name in classes_dict[cls_id]:
                classes_dict[cls_id][class_name] = classes_dict[cls_id][class_name] + 1
            else:
                classes_dict[cls_id][class_name] = 1
        else:
            classes_dict[cls_id] = {class_name:1}

        # cls_id = 0
        xmlbox = obj.find('bndbox')
        b = (float(xmlbox.find('xmin').text), float(xmlbox.find('ymin').text), float(xmlbox.find('xmax').text), float(xmlbox.find('ymax').text))
        bb = convert((width,height),b)
        list_file.write(str(cls_id) + " " + " ".join([str(a) for a in bb]) + '\n')

In [5]:
def check_and_create_folder(folder_path):
    if not os.path.exists(folder_path):
        os.makedirs(folder_path) 


def convert_xml_to_yolo(source_folder, target_folder):
    train_val = 0.8

    classes_dict = {}
    sample_list_ordered_by_class = {}

    train_yolo_image_folder = os.path.join(target_folder, "images\\train")
    val_yolo_image_folder = os.path.join(target_folder, "images\\val")

    train_yolo_label_folder = os.path.join(target_folder, "labels\\train")
    val_yolo_label_folder = os.path.join(target_folder, "labels\\val")

    check_and_create_folder(train_yolo_image_folder)
    check_and_create_folder(val_yolo_image_folder)
    check_and_create_folder(train_yolo_label_folder)
    check_and_create_folder(val_yolo_label_folder)

    for root, folders, files in os.walk(source_folder):
        for file in files:
            file_name,file_type = os.path.splitext(file)
            if not (file_type == ".xml" or file_type == ".txt"):
                voc_annotation_file = open(os.path.join(root,file_name+".xml"), encoding='utf-8')
                tree = ET.parse(voc_annotation_file)
                r = tree.getroot()
                temp_list = set([])
                for obj in r.iter('object'):
                    class_name = obj.find('name').text.upper()
                    if not class_name in category_map:
                        print(class_name)
                        class_name = "INSECTA"
                    temp_list.add(class_name)
                for c in temp_list:
                    if c in sample_list_ordered_by_class:
                        sample_list_ordered_by_class[c].append(file)
                    else:
                        sample_list_ordered_by_class[c] = [file]

    sample_list_ordered_by_class = dict(sorted(sample_list_ordered_by_class.items(), key=lambda item: len(item[1])))
    val_image_list = set([])
    train_image_list = set([])
    for c in sample_list_ordered_by_class.keys():
        val_list = sample(sample_list_ordered_by_class[c], int(max(1, len(sample_list_ordered_by_class[c])*(1-train_val))))
        train_list = set(sample_list_ordered_by_class[c])-set(val_list)
        val_image_list = val_image_list.union(val_list)
        train_image_list = train_image_list.union(train_list)

    for root, folders, files in os.walk(source_folder):
        for file in files:
            file_name,file_type = os.path.splitext(file)
            if not (file_type == ".xml" or file_type == ".txt"):
                if file in train_image_list:
                    shutil.copy(os.path.join(root,file),os.path.join(train_yolo_image_folder,file))

                    annotation_file = open(os.path.join(train_yolo_label_folder,file_name+".txt"), 'w', encoding='utf-8')
                    
                    convert_annotation(os.path.join(root,file_name+".xml"), annotation_file, classes_dict, os.path.join(root,file))

                    annotation_file.write('\n')

                    annotation_file.close()
                else:
                    shutil.copy(os.path.join(root,file),os.path.join(val_yolo_image_folder,file))

                    annotation_file = open(os.path.join(val_yolo_label_folder,file_name+".txt"), 'w', encoding='utf-8')

                    convert_annotation(os.path.join(root,file_name+".xml"), annotation_file, classes_dict, os.path.join(root,file))

                    annotation_file.write('\n')

                    annotation_file.close()


    print(classes_dict)

In [8]:
org_path = "F:\\pest_data\\Multitask_or_multimodality\\annotated_data"

yolo_path = "X:\\pervasive_group\\PestProject\\YOLO_07JAN25_ALL_INSECT"

convert_xml_to_yolo(org_path, yolo_path)

{1: {'11121': 4755, 'GRAIN APHID (SITOBION AVENAE)': 265, 'ROSE GRAIN APHID': 1, 'GRAIN APHID': 210}, 2: {'11122': 1570, 'BIRD CHERRY OAK APHIDS': 80, 'BIRD CHERRY OAT APHID': 214}, 0: {'11111': 151, 'INSECTA': 1963, 'FROGHOPPER (CERCOPIDAE)': 1, 'SCARABAEIDAE': 1}, 4: {'POLLEN BEETLE (MELIGETHES SPP.)': 10706}, 5: {'SNAIL': 82}, 6: {'CEREAL LEAF BEETLE (OULEMA MELANOPUS)': 172}, 7: {'FLY (DIPTERA)': 2112, 'BEAN SEED FLY (DELIA SPP.)': 63, 'FLY (MELANOSTOMA SPP.)': 58, 'BIBIONID FLY (BIBIONIDAE)': 43, 'MUSCIDAE (FLY)': 4, 'FUNGUS GNAT (MYCETOPHILIDAE)': 8, 'LEAF MINERS': 2}, 8: {'CABBAGE STEM FLEA BETTLE': 163}, 9: {'LADYBUG (COCCINELLIDAE)': 980}, 12: {'SPIDER (ARANEUS SPP.)': 102}, 14: {'BEETLE (COLEOPTERA)': 335}, 13: {'CHIRONOMID MIDGE': 175, 'CHIRONOMID MIDGE (MALE)': 6}, 15: {'MOSQUITO': 110}, 10: {'LADYBUG (COCCINELLIDAE) (PUPA)': 166}, 16: {'WASP': 43}, 18: {'SLUG': 82}, 19: {'CABBAGE WHITEFLY': 22}, 11: {'LADYBUG (COCCINELLIDAE) (LARVAE)': 11}, 20: {'HEMIPTERA (PLANT BUG)': 4}

In [21]:
import yaml

def dataset_statistics(dataset_yaml_file):
    with open(dataset_yaml_file) as file:
        dataset = yaml.load(file, Loader=yaml.FullLoader)
    
    category_map = dataset['names']
    
    dataset_path = dataset['path']

    train_anno_path = os.path.join(dataset_path, "labels\\train")
    val_anno_path = os.path.join(dataset_path, "labels\\val")

    print(train_anno_path)

    category_statistics = {}

    for root, folders, files in os.walk(train_anno_path):
        for file in files:
            file_name,file_type = os.path.splitext(file)
            if file_type == ".txt":
                annotation_file = open(os.path.join(root,file), encoding='utf-8')
                temp_list = set([])
                for line in annotation_file:
                    line = line.strip()
                    parts = line.split(' ')
                    if len(parts) > 2:
                        class_id = int(parts[0])
                        if class_id in category_statistics:
                            category_statistics[class_id][1] = category_statistics[class_id][1] + 1
                        else:
                            category_statistics[class_id] = [category_map[class_id], 1, 0]
                        temp_list.add(class_id)
                    
                for c in temp_list:
                    category_statistics[c][2] = category_statistics[c][2] + 1

    category_statistics = dict(sorted(category_statistics.items(), key=lambda item: item[0], reverse=True))
    print(f"Training Dataset Statistics {category_statistics}")

    val_statistics = {}

    for root, folders, files in os.walk(val_anno_path):
        for file in files:
            file_name,file_type = os.path.splitext(file)
            if file_type == ".txt":
                annotation_file = open(os.path.join(root,file), encoding='utf-8')
                temp_list = set([])
                for line in annotation_file:
                    line = line.strip()
                    parts = line.split(' ')
                    if len(parts) > 2:
                        class_id = int(parts[0])
                        if class_id in category_statistics:
                            category_statistics[class_id][1] = category_statistics[class_id][1] + 1
                        else:
                            category_statistics[class_id] = [category_map[class_id], 1, 0]

                        if class_id in val_statistics:
                            val_statistics[class_id][1] = val_statistics[class_id][1] + 1
                        else:
                            val_statistics[class_id] = [category_map[class_id], 1, 0]

                        temp_list.add(class_id)
                    
                for c in temp_list:
                    category_statistics[c][2] = category_statistics[c][2] + 1
                    val_statistics[c][2] = val_statistics[c][2] + 1

    val_statistics = dict(sorted(val_statistics.items(), key=lambda item: item[0], reverse=True))

    print(f"Validation Dataset Statistics {val_statistics}")
    print(f"Total Dataset Statistics {category_statistics}")


In [22]:
dataset_statistics('../uk_pest_dataset_06JAN25_all_insect.yaml')

F:\pest_data\Multitask_or_multimodality\YOLO_06JAN25_ALL_INSECT\labels\train
Training Dataset Statistics {27: ['LONGICORN', 5, 5], 26: ['PYRRHOCORIDAE', 6, 6], 25: ['ANT', 9, 9], 24: ['GROUND BETTLE (HARPALUS SPP)', 26, 26], 23: ['BUMBLEBEE', 88, 88], 22: ['EARTHWORM', 17, 17], 21: ['HEMIPTERA (PLANT BUG)', 3, 3], 20: ['CABBAGE WHITEFLY', 20, 11], 19: ['SLUG', 69, 54], 18: ['BEE', 219, 219], 17: ['WASP', 36, 36], 16: ['MOSQUITO', 89, 85], 15: ['BEETLE (COLEOPTERA)', 275, 229], 14: ['CHIRONOMID MIDGE', 153, 114], 13: ['SPIDER (ARANEUS SPP.)', 85, 85], 12: ['LADYBUG (COCCINELLIDAE) (LARVAE)', 9, 9], 11: ['LADYBUG (COCCINELLIDAE) (PUPA)', 137, 137], 10: ['LADYBUG (COCCINELLIDAE)', 800, 778], 9: ['CABBAGE STEM FLEA BETTLE', 135, 109], 8: ['FLY (DIPTERA)', 1899, 1817], 7: ['CEREAL LEAF BEETLE (OULEMA MELANOPUS)', 143, 117], 6: ['SNAIL', 67, 65], 5: ['POLLEN BEETLE (MELIGETHES SPP.)', 8780, 3021], 4: ['WILLOW CARROT APHID', 290, 17], 3: ['BIRD CHERRY OAK APHID', 1777, 318], 1: ['GRAIN APHID 